In [2]:
import pandas as pd
from tqdm import tqdm
import json

Big Picture Architecture (what I am building)

I am building a data generation pipeline:
1. Load my dataset
2. Analyze structure (patterns + distributions)
3. Build rule dictionaries (the interpretive logic)
4. Sample structured prompts
5. Use LLM to generate batches
6. Validate outputs (schema + logic)
7. Append + repeat until 500-1000 rows

CSV->Pattern Extraction->Rules->LLM Generator->Validator->Final Dataset

In [4]:
nfl_df = pd.read_csv("nfl_media.csv")
nfl_df.head()

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14
0,1.0,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN,NaN
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN,NaN
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN,NaN
3,4.0,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN,NaN
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN,NaN


In [5]:
nfl_df.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL', 'Unnamed: 13',
       'Unnamed: 14'],
      dtype='str')

In [6]:
nfl_df = nfl_df.drop(columns=['Unnamed: 13', 'Unnamed: 14'])

In [7]:
nfl_df.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL'],
      dtype='str')

In [8]:
nfl_df.shape

(263, 13)

In [9]:
nfl_cleaned = nfl_df.dropna()

In [10]:
nfl_cleaned.shape

(75, 13)

In [11]:
nfl_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 75 non-null     float64
 1   Player Name        75 non-null     str    
 2   Player Status      75 non-null     str    
 3   Year of Event      75 non-null     float64
 4   Statement Made     75 non-null     str    
 5   Event Description  75 non-null     str    
 6   Platform           75 non-null     str    
 7   Event Type         75 non-null     str    
 8   Media Tone         75 non-null     str    
 9   Post Status        75 non-null     str    
 10  Media Coverage     75 non-null     str    
 11  Sources            75 non-null     str    
 12  URL                75 non-null     str    
dtypes: float64(2), str(11)
memory usage: 34.7 KB


In [12]:
nfl_cleaned.dtypes

ID                   float64
Player Name              str
Player Status            str
Year of Event        float64
Statement Made           str
Event Description        str
Platform                 str
Event Type               str
Media Tone               str
Post Status              str
Media Coverage           str
Sources                  str
URL                      str
dtype: object

In [13]:
nfl_df['Event Type'].value_counts()

Event Type
Controversy    39
Support        16
Humor          10
Reflection      7
Apology         3
Name: count, dtype: int64

In [14]:
nfl_df['Media Tone'].value_counts()

Media Tone
Negative     33
Positive     25
Mixed        16
Positive      1
Name: count, dtype: int64

In [15]:
nfl_df['Platform'].value_counts().head()

Platform
Twitter/X                44
Instagram                10
St. Brown Podcast         1
"New Heights" podcast     1
"It's Giving" podcast     1
Name: count, dtype: int64

In [16]:
nfl_cleaned['Sources'].value_counts().head()

Sources
Yahoo Sports    11
ESPN             5
US Weekly        4
BroncosWire      2
MSN              2
Name: count, dtype: int64

In [17]:
nfl_cleaned.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL'],
      dtype='str')

In [18]:
nfl_cleaned.rename(columns={'ID': 'id', 'Player Name': 'player_name', 'Player Status': 'player_status', 'Year of Event': 'year_of_event', 
                                          'Statement Made': 'statement_made', 'Event Description': 'event_description', 'Platform':'platform',
                                          'Event Type': 'event_type', 'Media Tone': 'media_tone', 'Post Status': 'post_status', 
                                          'Media Coverage': 'media_coverage', 'Platform': 'platform', 'Sources': 'sources', 'URL': 'url'}, inplace=True)

In [19]:
{
  "ID": int,
  "Player Name": str,
  "Player Status": ["Playing", "Retired", "Free Agent"],
  "Year of Event": int,
  "Statement Made": str,
  "Event Description": str,
  "Platform": ["Twitter/X", "Instagram", "Podcast", "Press Conference", "Public Appearance"],
  "Event Type": ["Controversy", "Apology", "Support", "Reflection", "Humor"],
  "Media Tone": ["Positive", "Negative", "Mixed"],
  "Post Status": ["Active", "Deleted", "Available", "Unavailable"],
  "Media Coverage": ["Low", "Medium", "High"],
  "Sources": list[str]
}

{'ID': int,
 'Player Name': str,
 'Player Status': ['Playing', 'Retired', 'Free Agent'],
 'Year of Event': int,
 'Statement Made': str,
 'Event Description': str,
 'Platform': ['Twitter/X',
  'Instagram',
  'Podcast',
  'Press Conference',
  'Public Appearance'],
 'Event Type': ['Controversy', 'Apology', 'Support', 'Reflection', 'Humor'],
 'Media Tone': ['Positive', 'Negative', 'Mixed'],
 'Post Status': ['Active', 'Deleted', 'Available', 'Unavailable'],
 'Media Coverage': ['Low', 'Medium', 'High'],
 'Sources': list[str]}

In [20]:
nfl_cleaned.columns

Index(['id', 'player_name', 'player_status', 'year_of_event', 'statement_made',
       'event_description', 'platform', 'event_type', 'media_tone',
       'post_status', 'media_coverage', 'sources', 'url'],
      dtype='str')

In [21]:
nfl_cleaned["event_type"] = nfl_cleaned["event_type"].str.strip()
nfl_cleaned["media_tone"] = nfl_cleaned["media_tone"].str.strip()
nfl_cleaned["platform"] = nfl_cleaned["platform"].str.strip()
nfl_cleaned["player_status"] = nfl_cleaned["player_status"].str.strip()

In [22]:
event_types = ["Controversy", "Apology", "Support", "Reflection", "Humor"]
media_tones = ["Positive", "Negative", "Mixed"]
platform = ["Twitter/X", "Instagram", "Podcast", "Press Conference", "Public Appearance"]
player_status = ["Playing", "Retired", "Free Agent"]
media_coverage = ["Low", "Medium", "High"]
post_status = ["Active", "Deleted", "Available", "Unavailable"]

Creating dictionaries to generate patterns to follow in interpretative variables

In [8]:
event_type_rules = {
    "Controversy": ["criticized", "backlash", "accused", "under fire"],
    "Apology": ["apologized", "regret", "sorry", "mistake"],
    "Support": ["supported", "defended", "donated", "stood by"],
    "Reflection": ["career", "retirement", "looking back", "legacy"],
    "Humor": ["joked", "sarcastic", "funny", "playful"]
}

In [9]:
media_tone_rules = {
    "Positive": ["praised", "celebrated", "highlighted"],
    "Negative": ["criticized", "controversial", "under fire"],
    "Mixed": ["debated", "mixed reactions", "split opinions"]
}

In [10]:
media_coverage_rules = {
    "High": [
        "widely reported", "major outlets", "headline", "viral",
        "extensive coverage", "national attention", "breaking news"
    ],
    "Medium": [
        "reported by several outlets", "moderate attention",
        "covered by sports media", "notable coverage"
    ],
    "Low": [
        "briefly mentioned", "limited coverage",
        "few outlets", "minor attention", "local report"
    ]
}

In [11]:
high_tier_outlets = [
    "ESPN", "NFL Network", "Yahoo Sports", "Fox Sports", "CBS Sports"
]

mid_tier_outlets = [
    "Bleacher Report", "SB Nation", "Sports Illustrated"
]

low_tier_outlets = [
    "local news", "team website", "small blog"
]

In [23]:
def classify_coverage(row):
    text = (str(row["event_description"]) + " " + str(row["sources"])).lower()
    
    score = 0
    
    # keyword signals
    for word in media_coverage_rules["High"]:
        if word in text:
            score += 2
            
    for word in media_coverage_rules["Medium"]:
        if word in text:
            score += 1
            
    for word in media_coverage_rules["Low"]:
        if word in text:
            score -= 1
    
    # outlet signals
    for outlet in high_tier_outlets:
        if outlet.lower() in text:
            score += 2
            
    for outlet in mid_tier_outlets:
        if outlet.lower() in text:
            score += 1
            
    # event-type boost (optional but realistic)
    if row["event_type"] == "Controversy":
        score += 1
    
    # final classification
    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"

In [28]:
platform_dict = {
    "Twitter/X": ["tweet", "X post", "retweeted"],
    "Instagram": ["Instagram post", "story", "caption"],
    "Podcast": ["podcast", "interview episode"],
    "Press Conference": ["press conference", "media availability"],
    "Public Appearance": ["event", "appearance", "ceremony"]
}

Get seed examples

In [30]:
seed_examples = nfl_cleaned.sample(8).to_dict(orient='records')
print(seed_examples[0])

{'id': 66.0, 'player_name': 'Josh Allen', 'player_status': 'Playing', 'year_of_event': 2023.0, 'statement_made': "I saw some stuff on Twitter and people should not be attacking him whatsoever. I'm glad that Damar's family came out and said that. ", 'event_description': 'Comes to the defense of Tee Higgins, who received hate from tackle leading to Hamlin to being hospitalized', 'platform': 'Post-practice news conference', 'event_type': 'Support', 'media_tone': 'Positive', 'post_status': 'Available', 'media_coverage': 'Low', 'sources': 'Cincinnati Enquirer', 'url': 'https://www.cincinnati.com/story/sports/nfl/bengals/2023/01/05/josh-allen-people-should-not-be-attacking-tee-higgins-whatsoever-as-damar-hamlin-recovers-cincinnati/69782959007/?gnt-cfr=1&gca-cat=p&gca-uir=true&gca-epti=z115240e1162xxv115240d--58--b--58--&gca-ft=200&gca-ds=sophi'}


-----

In [31]:
def build_prompt(seed_examples_batch):
    return f"""
You are generating structured NFL media dataset records.

TASK:
Generate 20 NEW rows of a dataset called nfl_media.

You MUST follow this schema exactly:
- ID (integer)
- Player Name (NFL player)
- Player Status (Playing, Retired, Free Agent)
- Year of Event (2020–2026)
- Statement Made (realistic quote or paraphrased statement)
- Event Description (context of event)
- Platform (Twitter/X, Instagram, Podcast, Press Conference, Public Appearance)
- Event Type (Controversy, Apology, Support, Reflection, Humor)
- Media Tone (Positive, Negative, Mixed)
- Post Status (Active, Deleted, Available, Unavailable)
- Media Coverage (Low, Medium, High)
- Sources (sports/media outlets)
- URL (realistic placeholder)

RULES:
- Be realistic (NFL-related only)
- Do NOT repeat identical events
- Maintain diversity in players and platforms
- Event Type must match Statement tone
- Media Tone must match framing of event
- Avoid exaggerated or fictional drama
- Keep consistent sports journalism style

EVENT TYPE RULES:
Controversy → criticism, backlash, accusations
Apology → remorse, apology, regret
Support → defending others, solidarity
Reflection → career thoughts, retirement, past review
Humor → joking, sarcasm, playful remarks

MEDIA TONE RULES:
Positive → praise, celebration
Negative → criticism, backlash framing
Mixed → balanced reporting

HERE ARE REAL EXAMPLES FROM MY DATASET:
{seed_examples_batch}

OUTPUT FORMAT:
Return ONLY valid JSON list of 20 objects.

Do NOT include explanations.
"""

In [32]:
prompt = build_prompt(seed_examples)
print(prompt)


You are generating structured NFL media dataset records.

TASK:
Generate 20 NEW rows of a dataset called nfl_media.

You MUST follow this schema exactly:
- ID (integer)
- Player Name (NFL player)
- Player Status (Playing, Retired, Free Agent)
- Year of Event (2020–2026)
- Statement Made (realistic quote or paraphrased statement)
- Event Description (context of event)
- Platform (Twitter/X, Instagram, Podcast, Press Conference, Public Appearance)
- Event Type (Controversy, Apology, Support, Reflection, Humor)
- Media Tone (Positive, Negative, Mixed)
- Post Status (Active, Deleted, Available, Unavailable)
- Media Coverage (Low, Medium, High)
- Sources (sports/media outlets)
- URL (realistic placeholder)

RULES:
- Be realistic (NFL-related only)
- Do NOT repeat identical events
- Maintain diversity in players and platforms
- Event Type must match Statement tone
- Media Tone must match framing of event
- Avoid exaggerated or fictional drama
- Keep consistent sports journalism style

EVENT

In [33]:
with open("batch1.json") as f:
    batch1 = json.load(f)

batch_df = pd.DataFrame(batch1)

batch_df.head()

,id,player_name,player_status,year_of_event,statement_made,event_description,platform,event_type,media_tone,post_status,media_coverage,sources,url
0,101,Patrick Mahomes,Playing,2024,"We didn't execute the way we expect to, and th...",Postgame comments after a regular season loss ...,Press Conference,Reflection,Mixed,Available,High,"ESPN, NFL Network",https://www.espn.com/nfl/story/_/id/39186641/c...
1,102,Jalen Hurts,Playing,2023,"It's about how we respond, not what happens.",Comments following a tough loss emphasizing te...,Press Conference,Reflection,Positive,Available,High,"NBC Sports, ESPN",https://www.nbcsports.com/nfl/profootballtalk/...
2,103,Odell Beckham Jr.,Playing,2021,I just want to be somewhere I'm appreciated an...,Social media post during tensions with the Cle...,Instagram,Controversy,Negative,Deleted,High,"Bleacher Report, ESPN",https://www.espn.com/nfl/story/_/id/32543974/c...
3,104,Russell Wilson,Playing,2022,I take full responsibility for how I played to...,Addressing criticism after a poor performance ...,Press Conference,Apology,Mixed,Available,High,"Fox Sports, ESPN",https://www.foxsports.com/stories/nfl/russell-...
4,105,Travis Kelce,Playing,2024,"Man, I gotta stop celebrating like I'm 21 ðŸ˜‚",Joking about his touchdown celebrations after ...,Podcast,Humor,Positive,Available,Medium,"New Heights Podcast, Yahoo Sports",https://www.youtube.com/@newheightshow


In [34]:
def validate_row(row):
    return (
        row["event_type"] in ["Controversy","Apology","Support","Reflection","Humor"]
        and row["media_tone"] in ["Positive","Negative","Mixed"]
        and row["media_coverage"] in ["Low","Medium","High"]
    )

batch_df["valid"] = batch_df.apply(validate_row, axis=1)

print(batch_df["valid"].value_counts())

valid
True    20
Name: count, dtype: int64


In [35]:
nfl_combined = pd.concat([nfl_cleaned, batch_df], ignore_index=True)

print(nfl_combined.shape)

(95, 14)


In [ ]:
all_batches = [nfl_combined]

for i in tqdm(range(1, 30)):  # ~30 batches
    
    input(f"\nGenerate batch {i+1} in ChatGPT and save as batch{i+1}.json, then press Enter...")
    
    with open(f"batch{i+1}.json", encoding="utf-8") as f:
        batch = json.load(f)
    
    batch_df = pd.DataFrame(batch)
    
    # validate
    batch_df = batch_df[
        batch_df["event_type"].isin(["Controversy","Apology","Support","Reflection","Humor"])
        & batch_df["media_tone"].isin(["Positive","Negative","Mixed"])
        & batch_df["media_coverage"].isin(["Low","Medium","High"])
    ]
    
    all_batches.append(batch_df)

final_df = pd.concat(all_batches, ignore_index=True)

  0%|          | 0/29 [00:00<?, ?it/s]

In [ ]:
final_df = final_df.drop_duplicates(subset=["statement_made"])

final_df = final_df.reset_index(drop=True)
final_df['id'] = range(1, len(final_df) + 1)

final_df.to_csv("nfl_media_expanded.csv", index=False)

print(final_df.shape)

(665, 14)


In [3]:
final_df = pd.read_csv("nfl_media_expanded.csv")

In [4]:
final_df.head()

,id,player_name,player_status,year_of_event,statement_made,event_description,platform,event_type,media_tone,post_status,media_coverage,sources,url,valid
0,1,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN
1,2,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN
2,3,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN
3,4,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN
4,5,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN


In [5]:
final_df['id'].duplicated().sum()

np.int64(0)

Interpretative portion - will use the dictionaries I created to label the interpretative variables and compare to AI's inputs

In [6]:
def classify_event_type(text):
    text = str(text).lower()
    
    for label, keywords in event_type_rules.items():
        if any(k in text for k in keywords):
            return label
    
    return "Reflection"

In [13]:
final_df["event_type_rule"] = final_df["statement_made"].apply(classify_event_type)

In [15]:
final_df["event_type"].value_counts()

event_type
Support        319
Reflection     154
Controversy    110
Humor           54
Apology         28
Name: count, dtype: int64

In [14]:
final_df["event_type_rule"].value_counts()

event_type_rule
Reflection    657
Apology         8
Name: count, dtype: int64

In [16]:
def classify_media_tone(text):
    text = str(text).lower()
    
    for label, keywords in media_tone_rules.items():
        if any(k in text for k in keywords):
            return label
    
    return "Mixed"

In [19]:
final_df["media_tone_rule"] = final_df["statement_made"].apply(classify_media_tone)

In [21]:
final_df['media_tone'].value_counts()

media_tone
Positive    473
Mixed       107
Negative     85
Name: count, dtype: int64

In [20]:
final_df['media_tone_rule'].value_counts()

media_tone_rule
Mixed       664
Negative      1
Name: count, dtype: int64

Have to refine the event type rules and media tone rules - maybe look at the common words used in the 'text' variable

In [29]:
final_df.columns

Index(['id', 'player_name', 'player_status', 'year_of_event', 'statement_made',
       'event_description', 'platform', 'event_type', 'media_tone',
       'post_status', 'media_coverage', 'sources', 'url', 'valid',
       'event_type_rule', 'media_type_rule', 'media_tone_rule',
       'media_coverage_rule'],
      dtype='str')

In [33]:
final_df['player_name'].value_counts().head()

player_name
Travis Kelce       7
Patrick Mahomes    7
Tyreek Hill        7
Caleb Williams     6
Saquon Barkley     6
Name: count, dtype: int64

In [30]:
final_df['statement_made']

0      Man I love the 4th...we all share in common th...
1                                 Texas is Fake Football
2      I deeply apologize...I do not stand for any fo...
3      happy easter...#shoutout to Jesus for takin on...
4           He will be a top 5 pick. Where yo son going?
                             ...                        
660    I’m here to play physical football. The Bills ...
661    I’m looking forward to learning from the guys ...
662    When the work is put in, the decisions speak f...
663    I fit this defense. Chicago wants toughness an...
664    Grateful for the chance through the Internatio...
Name: statement_made, Length: 665, dtype: str

In [24]:
final_df["media_coverage_rule"] = final_df.apply(classify_coverage, axis=1)

In [26]:
final_df['media_coverage'].value_counts()

media_coverage
Medium    347
High      194
Low       124
Name: count, dtype: int64

In [25]:
final_df["media_coverage_rule"].value_counts()

media_coverage_rule
Low       349
Medium    184
High      132
Name: count, dtype: int64